# 04 · Refusal feature selection

Runs Llama 3 8B Instruct on the `target` (harmful AdvBench prompts) and `retain` (harmless instructions) corpora, encodes layer-12 activations with the K=56 SAE and ranks latents that fire on harmful prompts but almost never on harmless ones (retain firing rate ≤ 0.05). The top-4 latents are saved to `selected_refusal_features_strict.json` and used by all later notebooks.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')

HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print('HF_TOKEN is not set in Colab secrets. Run login() manually if the model is gated.')

In [ ]:
from __future__ import annotations

from pathlib import Path
from contextlib import contextmanager
from typing import Dict, Iterator, List, Optional, Tuple

import gc
import json
import random
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)

In [ ]:
MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'
TEXT_CORPUS_DIR = Path('/content/drive/MyDrive/Diploma/corpora2')
ACTIVATION_DATA_DIR = Path('/content/drive/MyDrive/Diploma/activation_datasets2')

SAE_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_runs/k56')
CKPT_DIR = SAE_OUT_DIR / 'checkpoints'
SAE_CKPT_PATH = CKPT_DIR / 'sae_epoch_025.pt'

EVAL_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_eval_k56_refusal_features')
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_OUT_DIR = EVAL_OUT_DIR / 'refusal_features_strict_retain005'
FEATURE_OUT_DIR.mkdir(parents=True, exist_ok=True)

D_IN = 4096
D_SAE = 16384
K = 56
HOOK_LAYER = 12

SAE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

SELECTED_FEATURES_JSON_PATH = FEATURE_OUT_DIR / 'selected_refusal_features_strict.json'
FEATURE_RANKING_CSV_PATH = FEATURE_OUT_DIR / 'refusal_feature_ranking_strict.csv'
RETAIN_FEATURE_STATS_PATH = FEATURE_OUT_DIR / 'retain_feature_stats.pt'
TARGET_FEATURE_STATS_PATH = FEATURE_OUT_DIR / 'target_feature_stats.pt'

print('SAE_CKPT_PATH:', SAE_CKPT_PATH)
print('EVAL_OUT_DIR:', EVAL_OUT_DIR)
print('FEATURE_OUT_DIR:', FEATURE_OUT_DIR)
print('SAE_DTYPE:', SAE_DTYPE)
print('MODEL_DTYPE:', MODEL_DTYPE)

JSONL_FEATURE_BATCH_SIZE = 4
JSONL_FEATURE_MAX_LENGTH = 512
FEATURE_MAX_RETAIN_RATE = 0.05
FEATURE_MIN_TARGET_COUNT = 10
FEATURE_MIN_TARGET_RATE_MODE = 'mean'
FEATURE_MIN_TARGET_RATE_MANUAL = 0.0
FEATURE_MIN_TARGET_RATE_MULTIPLIER = 1.0
FEATURE_TOP_N = 4

In [ ]:
def read_jsonl(path: Path) -> List[dict]:
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def batch_iter(items: List, batch_size: int) -> Iterator[List]:
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_hidden_from_output(output):
    return output[0] if isinstance(output, tuple) else output


def replace_hidden_in_output(output, new_hidden):
    if isinstance(output, tuple):
        output = list(output)
        output[0] = new_hidden
        return tuple(output)
    return new_hidden

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_in: int, d_sae: int, k: int, dtype=torch.float16):
        super().__init__()
        self.d_in = d_in
        self.d_sae = d_sae
        self.k = k
        self.W_enc = nn.Parameter(torch.empty(d_in, d_sae, dtype=dtype))
        self.W_dec = nn.Parameter(torch.empty(d_sae, d_in, dtype=dtype))
        self.b_enc = nn.Parameter(torch.zeros(d_sae, dtype=dtype))
        self.b_dec = nn.Parameter(torch.zeros(d_in, dtype=dtype))

    def encode_topk(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        pre_acts = (x - self.b_dec) @ self.W_enc + self.b_enc
        topk_vals, topk_idx = pre_acts.topk(self.k, dim=-1)
        return F.relu(topk_vals), topk_idx

    def decode_topk(self, topk_vals: torch.Tensor, topk_idx: torch.Tensor) -> torch.Tensor:
        selected_dec = self.W_dec[topk_idx]
        return (topk_vals.unsqueeze(-1) * selected_dec).sum(dim=1) + self.b_dec

    def encode_dense(self, x: torch.Tensor) -> torch.Tensor:
        topk_vals, topk_idx = self.encode_topk(x)
        feat = torch.zeros(x.shape[0], self.d_sae, device=x.device, dtype=x.dtype)
        feat.scatter_(-1, topk_idx, topk_vals)
        return feat

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        topk_vals, topk_idx = self.encode_topk(x)
        x_recon = self.decode_topk(topk_vals, topk_idx)
        return {
            'x_recon': x_recon,
            'mse': F.mse_loss(x_recon.float(), x.float()),
            'l0': (topk_vals > 0).float().sum(dim=-1).mean(),
            'topk_vals': topk_vals,
            'topk_idx': topk_idx,
        }


def load_sae() -> TopKSAE:
    sae = TopKSAE(D_IN, D_SAE, K, dtype=SAE_DTYPE).to(DEVICE)
    ckpt = torch.load(SAE_CKPT_PATH, map_location=DEVICE, weights_only=True)
    sae.load_state_dict(ckpt['model_state_dict'])
    sae.eval()
    print('Loaded SAE:', SAE_CKPT_PATH)
    print('checkpoint epoch:', ckpt.get('epoch'), 'step:', ckpt.get('step'))
    return sae

sae = load_sae()

In [ ]:
print('Loading tokenizer and Llama 3 8B Instruct...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
    device_map='auto',
    low_cpu_mem_usage=True,
    attn_implementation='sdpa',
)
model.eval()
INPUT_DEVICE = next(model.model.embed_tokens.parameters()).device
print('INPUT_DEVICE:', INPUT_DEVICE)

In [ ]:
def render_chat(user_prompt: str, system_prompt: str = 'You are a helpful assistant.') -> str:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def render_record_to_text(record: dict) -> str:
    return tokenizer.apply_chat_template(record['messages'], tokenize=False, add_generation_prompt=True)


def tokenize_texts(texts: List[str], max_length: int = 512) -> Dict[str, torch.Tensor]:
    batch = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=max_length,
        add_special_tokens=False,
    )
    return {
        'input_ids': batch['input_ids'].to(INPUT_DEVICE),
        'attention_mask': batch['attention_mask'].to(INPUT_DEVICE),
    }

In [ ]:
class StopForwardAfterHook(Exception):
    pass


@torch.inference_mode()
def collect_feature_stats_from_jsonl(
    jsonl_path: Path,
    batch_size: int = JSONL_FEATURE_BATCH_SIZE,
    max_length: int = JSONL_FEATURE_MAX_LENGTH,
    save_path: Optional[Path] = None,
    force_recompute: bool = False,
) -> Dict[str, torch.Tensor]:
    if save_path is not None and save_path.exists() and not force_recompute:
        return torch.load(save_path, map_location='cpu', weights_only=True)

    records = read_jsonl(jsonl_path)
    feature_counts = torch.zeros(D_SAE, dtype=torch.long)
    feature_act_sums = torch.zeros(D_SAE, dtype=torch.float64)
    total_tokens = 0

    def hook(module, inputs, output):
        hidden = get_hidden_from_output(output)
        captured['hidden'] = hidden.detach()
        raise StopForwardAfterHook()

    for batch_records in tqdm(list(batch_iter(records, batch_size)), desc=jsonl_path.name):
        texts = [render_record_to_text(r) for r in batch_records]
        batch = tokenize_texts(texts, max_length=max_length)
        captured = {}
        handle = model.model.layers[HOOK_LAYER].register_forward_hook(hook)
        try:
            try:
                model.model(
                    input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'],
                    use_cache=False,
                )
            except StopForwardAfterHook:
                pass
        finally:
            handle.remove()

        hidden = captured['hidden']
        mask = batch['attention_mask'].to(hidden.device).bool()
        x = hidden[mask].to(device=DEVICE, dtype=SAE_DTYPE)
        out = sae(x)
        active_mask = out['topk_vals'] > 0
        active_idx = out['topk_idx'][active_mask].detach().cpu().long()
        active_vals = out['topk_vals'][active_mask].detach().float().cpu().double()

        if active_idx.numel() > 0:
            feature_counts.scatter_add_(0, active_idx, torch.ones_like(active_idx, dtype=torch.long))
            feature_act_sums.scatter_add_(0, active_idx, active_vals)
        total_tokens += int(x.shape[0])
        del batch, captured, hidden, mask, x, out, active_idx, active_vals
        cleanup_cuda()

    stats = {
        'jsonl_path': str(jsonl_path),
        'total_tokens': torch.tensor(total_tokens, dtype=torch.long),
        'feature_counts': feature_counts,
        'feature_act_sums': feature_act_sums,
        'firing_rate': feature_counts.float() / max(total_tokens, 1),
        'mean_active_value': feature_act_sums / feature_counts.clamp(min=1).double(),
    }
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(stats, save_path)
    return stats

In [ ]:
def resolve_min_target_rate(target_rate: torch.Tensor) -> float:
    if FEATURE_MIN_TARGET_RATE_MODE == 'manual':
        return float(FEATURE_MIN_TARGET_RATE_MANUAL)
    return float(target_rate.float().mean().item() * FEATURE_MIN_TARGET_RATE_MULTIPLIER)


def rank_refusal_features(retain_stats: Dict[str, torch.Tensor], target_stats: Dict[str, torch.Tensor]) -> pd.DataFrame:
    eps = 1e-9
    retain_rate = retain_stats['firing_rate'].float()
    target_rate = target_stats['firing_rate'].float()
    retain_count = retain_stats['feature_counts'].long()
    target_count = target_stats['feature_counts'].long()
    target_mean_value = target_stats['mean_active_value'].float()
    retain_mean_value = retain_stats['mean_active_value'].float()
    min_target_rate = resolve_min_target_rate(target_rate)

    mask = (
        (retain_rate <= FEATURE_MAX_RETAIN_RATE) &
        (target_count >= FEATURE_MIN_TARGET_COUNT) &
        (target_rate > retain_rate) &
        (target_rate >= min_target_rate)
    )

    score = (
        target_rate
        * torch.log1p(target_count.float())
        * torch.log1p((target_rate + eps) / (retain_rate + eps))
    )

    rows = []
    for j in torch.nonzero(mask, as_tuple=False).flatten().tolist():
        rows.append({
            'feature': int(j),
            'score': float(score[j].item()),
            'retain_firing_rate': float(retain_rate[j].item()),
            'target_firing_rate': float(target_rate[j].item()),
            'score_diff': float((target_rate[j] - retain_rate[j]).item()),
            'score_ratio': float(((target_rate[j] + eps) / (retain_rate[j] + eps)).item()),
            'retain_count': int(retain_count[j].item()),
            'target_count': int(target_count[j].item()),
            'retain_mean_active_value': float(retain_mean_value[j].item()),
            'target_mean_active_value': float(target_mean_value[j].item()),
        })

    return pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)


def save_selected_features(ranking_df: pd.DataFrame, top_n: int = FEATURE_TOP_N) -> List[int]:
    selected = ranking_df.head(top_n)['feature'].astype(int).tolist()
    payload = {
        'selected_features': selected,
        'top_n': top_n,
        'ranking_csv': str(FEATURE_RANKING_CSV_PATH),
        'policy': {
            'max_retain_rate': FEATURE_MAX_RETAIN_RATE,
            'min_target_count': FEATURE_MIN_TARGET_COUNT,
            'min_target_rate_mode': FEATURE_MIN_TARGET_RATE_MODE,
            'min_target_rate_multiplier': FEATURE_MIN_TARGET_RATE_MULTIPLIER,
        },
    }
    write_json(SELECTED_FEATURES_JSON_PATH, payload)
    return selected

In [ ]:
FORCE_RECOMPUTE_FEATURE_STATS = False

retain_stats = collect_feature_stats_from_jsonl(
    TEXT_CORPUS_DIR / 'retain.jsonl',
    save_path=RETAIN_FEATURE_STATS_PATH,
    force_recompute=FORCE_RECOMPUTE_FEATURE_STATS,
)
target_stats = collect_feature_stats_from_jsonl(
    TEXT_CORPUS_DIR / 'target.jsonl',
    save_path=TARGET_FEATURE_STATS_PATH,
    force_recompute=FORCE_RECOMPUTE_FEATURE_STATS,
)

ranking_df = rank_refusal_features(retain_stats, target_stats)
ranking_df.to_csv(FEATURE_RANKING_CSV_PATH, index=False)
selected_features = save_selected_features(ranking_df, top_n=FEATURE_TOP_N)
print('Selected features:', selected_features)
display(ranking_df.head(30))

In [ ]:
def plot_retain_vs_target(ranking_df: pd.DataFrame, retain_stats, target_stats, path: Path, selected: List[int], zoom: bool = False):
    retain_rate = retain_stats['firing_rate'].float().numpy()
    target_rate = target_stats['firing_rate'].float().numpy()

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(retain_rate, target_rate, s=10, alpha=0.25, label='all features')

    if selected:
        selected = [int(x) for x in selected]
        ax.scatter(retain_rate[selected], target_rate[selected], s=60, label='selected')
        for j in selected:
            ax.annotate(str(j), (retain_rate[j], target_rate[j]), fontsize=9)

    ax.axline((0, 0), slope=1, linestyle='--', linewidth=1, label='target = retain')
    ax.axvline(FEATURE_MAX_RETAIN_RATE, linestyle=':', linewidth=1, label=f'retain = {FEATURE_MAX_RETAIN_RATE}')
    ax.axhline(resolve_min_target_rate(target_stats['firing_rate']), linestyle=':', linewidth=1, label='target threshold')
    ax.set_xlabel('retain firing rate')
    ax.set_ylabel('target firing rate')
    ax.set_title('SAE feature firing rates: retain vs target')
    if zoom:
        ax.set_xlim(-0.005, FEATURE_MAX_RETAIN_RATE * 1.5)
        ax.set_ylim(-0.005, max(0.4, target_rate[selected].max() * 1.2 if selected else 0.4))
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()

plot_retain_vs_target(
    ranking_df,
    retain_stats,
    target_stats,
    FEATURE_OUT_DIR / 'retain_vs_target_scatter_strict.png',
    selected_features,
    zoom=False,
)
plot_retain_vs_target(
    ranking_df,
    retain_stats,
    target_stats,
    FEATURE_OUT_DIR / 'retain_vs_target_scatter_strict_zoom.png',
    selected_features,
    zoom=True,
)

In [ ]:
selected_table = ranking_df[ranking_df['feature'].isin(selected_features)].copy()
selected_table.to_csv(FEATURE_OUT_DIR / 'selected_refusal_features_table.csv', index=False)
display(selected_table)